In [1]:
import openstudio
import subprocess
import os
import shutil





In [1]:
import os
import shutil
import subprocess
import openstudio
import pandas as pd

def check_epw_quality(epw_path):
    """
    Perform quality checks on an EPW file and return a summary of whether each variable is 'Good' or 'Suspicious'.
    
    Parameters:
        epw_path (str): Path to the EPW file.
    
    Returns:
        dict: Dictionary with quality check results for each tested variable.
    """
    # Read EPW file (skip header, start from line 9)
    epw_df = pd.read_csv(epw_path, skiprows=8, header=None)

    # Dictionary to store check results
    quality_checks = {}

    ### 1️⃣ Check for Missing Data ###
    missing_values = epw_df.isnull().sum().sum()
    quality_checks["Missing Data"] = "Suspicious" if missing_values > 0 else "Good"

    ### 2️⃣ Enhanced Dry Bulb Temperature Checks ###
    dry_bulb_temp = epw_df[6]
    month = epw_df[1]

    # Extreme values (-50°C to 60°C)
    extreme_values = ((dry_bulb_temp < -50) | (dry_bulb_temp > 60)).any()
    quality_checks["Extreme Temperature"] = "Suspicious" if extreme_values else "Good"

    # Sudden jumps (> 15°C per hour)
    temp_diff = dry_bulb_temp.diff().abs()
    rapid_jumps = (temp_diff > 15).any()
    quality_checks["Sudden Temp Jumps"] = "Suspicious" if rapid_jumps else "Good"

    # Constant temperature for more than 12 hours
    const_periods = (dry_bulb_temp.rolling(window=12, min_periods=1).std() == 0).any()
    quality_checks["Constant Temp Periods"] = "Suspicious" if const_periods else "Good"

    # Unrealistic day-night swings (<3°C or >30°C)
    epw_df["daily_max"] = dry_bulb_temp.groupby(epw_df[2]).transform("max")
    epw_df["daily_min"] = dry_bulb_temp.groupby(epw_df[2]).transform("min")
    epw_df["daily_range"] = epw_df["daily_max"] - epw_df["daily_min"]
    unrealistic_swings = ((epw_df["daily_range"] < 3) | (epw_df["daily_range"] > 30)).any()
    quality_checks["Day-Night Swings"] = "Suspicious" if unrealistic_swings else "Good"

    # Seasonal temperature mismatches
    summer_issues = ((month.isin([6, 7, 8])) & (dry_bulb_temp < 0)).any()  # Summer months with freezing temps
    winter_issues = ((month.isin([12, 1, 2])) & (dry_bulb_temp > 40)).any()  # Winter months with extreme heat
    quality_checks["Summer Freezing"] = "Suspicious" if summer_issues else "Good"
    quality_checks["Winter Extreme Heat"] = "Suspicious" if winter_issues else "Good"

    # Missing temperature data
    missing_temps = dry_bulb_temp.isna().any()
    quality_checks["Missing Temperature Data"] = "Suspicious" if missing_temps else "Good"

    return quality_checks

def run_energyplus_simulations(year):
    input_csv = f'resources/zip_code_list_{year}.csv'
    # Load the input CSV.
    df = pd.read_csv(input_csv)
    
    # Add the "EnergyPlus Status" column if it doesn't exist.
    if "EnergyPlus Status" not in df.columns:
        df["EnergyPlus Status"] = ""
    
    # Ensure quality check columns exist.
    quality_columns = [
        "Missing Data",
        "Extreme Temperature",
        "Sudden Temp Jumps",
        "Constant Temp Periods",
        "Day-Night Swings",
        "Summer Freezing",
        "Winter Extreme Heat",
        "Missing Temperature Data"
    ]
    for col in quality_columns:
        if col not in df.columns:
            df[col] = ""
    
    # Get the unique weather station IDs.
    unique_stations = df[f"weather_station_wmo_{year}"].unique()

    # Set EnergyPlus installation location.
    energyplus_path = '/Applications/EnergyPlus-23-2-0/energyplus'
    current_directory = os.getcwd()


    print('***************')
    print(len(unique_stations))
    print('***************')

    k = 0
    for station in unique_stations:
        print(k)

        # If any row with this weather station already has a non-empty EnergyPlus Status, skip simulation.
        station_rows = df[df[f"weather_station_wmo_{year}"] == station]
        if station_rows["EnergyPlus Status"].notna().all() and (station_rows["EnergyPlus Status"] != "").all():
            k += 1
            continue

        epw_path = os.path.join(f"epws_wmo_{year}", f'{station}_{year}.epw')

        # Create (or reuse) the simulation directory.
        out_dir = "test_epws_dir"
        os.makedirs(out_dir, exist_ok=True)

        min_idf = openstudio.IdfFile().load("resources/min.idf").get()
        # Save IDF file
        min_idf.save(f"{out_dir}/in.idf", True)

        # Copy the minimal IDF file and the EPW file into the simulation directory.
        shutil.copy(epw_path, os.path.join(out_dir, "in.epw"))

        # Change directory to the simulation folder.
        os.chdir(out_dir)

        # Run EnergyPlus simulation.
        process = subprocess.run(
            [energyplus_path, "-w", "in.epw", "in.idf"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        
        # Check if the simulation ran successfully.
        status = "Good" if process.stderr.strip() == "EnergyPlus Completed Successfully." else "Bad"

        # Return to the original directory.
        os.chdir(current_directory)

        # Update simulation status for all rows corresponding to this weather station.
        df.loc[df[f"weather_station_wmo_{year}"] == station, "EnergyPlus Status"] = status

        # Run quality checks on the EPW file.
        quality_results = check_epw_quality(epw_path)
        for key, value in quality_results.items():
            df.loc[df[f"weather_station_wmo_{year}"] == station, key] = value

        # Save and reload CSV after processing each station.
        df.to_csv(input_csv, index=False)
        df = pd.read_csv(input_csv)

        k += 1

    return df

year = 2023
# Run the EnergyPlus simulations, perform EPW quality checks, and print the updated DataFrame.
final_df = run_energyplus_simulations(year)
final_df


***************
2573
***************
0
[utilities.idf.IdfObject] <1> IdfObject of type 'OutputControl:Files' cannot have field index of 31. Cutting off IdfObject field parsing here, with the following text remaining: 
No


/var/folders/cj/vhw1_9g1151dh76sjlgdyq19k7f3h5/T/ipykernel_60650/3070656699.py:149: DtypeWarning: Columns (30,31,32,33,34,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


1
[utilities.idf.IdfObject] <1> IdfObject of type 'OutputControl:Files' cannot have field index of 31. Cutting off IdfObject field parsing here, with the following text remaining: 
No


/var/folders/cj/vhw1_9g1151dh76sjlgdyq19k7f3h5/T/ipykernel_60650/3070656699.py:149: DtypeWarning: Columns (30,31,32,33,34,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


2
[utilities.idf.IdfObject] <1> IdfObject of type 'OutputControl:Files' cannot have field index of 31. Cutting off IdfObject field parsing here, with the following text remaining: 
No


...

/var/folders/cj/vhw1_9g1151dh76sjlgdyq19k7f3h5/T/ipykernel_60650/3070656699.py:149: DtypeWarning: Columns (30,31,32,33,34,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


8
[utilities.idf.IdfObject] <1> IdfObject of type 'OutputControl:Files' cannot have field index of 31. Cutting off IdfObject field parsing here, with the following text remaining: 
No


/var/folders/cj/vhw1_9g1151dh76sjlgdyq19k7f3h5/T/ipykernel_60650/3070656699.py:149: DtypeWarning: Columns (30,31,32,33,34,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


9
[utilities.idf.IdfObject] <1> IdfObject of type 'OutputControl:Files' cannot have field index of 31. Cutting off IdfObject field parsing here, with the following text remaining: 
No
10
[utilities.idf.IdfObject] <1> IdfObject of type 'OutputControl:Files' cannot have field index of 31. Cutting off IdfObject field parsing here, with the following text remaining: 
No
11
[utilities.idf.IdfObject] <1> IdfObject of type 'OutputControl:Files' cannot have field index of 31. Cutting off IdfObject field parsing here, with the following text remaining: 
No
12
[utilities.idf.IdfObject] <1> IdfObject of type 'OutputControl:Files' cannot have field index of 31. Cutting off IdfObject field parsing here, with the following text remaining: 
No
13
[utilities.idf.IdfObject] <1> IdfObject of type 'OutputControl:Files' cannot have field index of 31. Cutting off IdfObject field parsing here, with the following text remaining: 
No
14
[utilities.idf.IdfObject] <1> IdfObject of type 'OutputControl:Files' can

,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,...,RH_holes_2023,EnergyPlus Status,Missing Data,Extreme Temperature,Sudden Temp Jumps,Constant Temp Periods,Day-Night Swings,Summer Freezing,Winter Extreme Heat,Missing Temperature Data
0,99638,52.89995,-168.93738,Nikolski,AK,Alaska,True,NaN,26,0.5,...,False,Bad,Good,Good,Good,Suspicious,Good,Good,Good,Good
1,99923,55.97796,-130.03671,Hyder,AK,Alaska,True,NaN,15,0.4,...,False,Good,Good,Good,Good,Good,Suspicious,Good,Good,Good
2,99776,63.38825,-143.41140,Tanacross,AK,Alaska,True,NaN,141,2.6,...,False,Good,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good
3,99566,61.28097,-142.78996,Chitina,AK,Alaska,True,NaN,98,0.0,...,False,Good,Good,Good,Suspicious,Good,Suspicious,Suspicious,Good,Good
4,99780,63.19468,-143.07797,Tok,AK,Alaska,True,NaN,1559,0.5,...,False,Good,Good,Good,Suspicious,Good,Suspicious,Suspicious,Good,Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32961,36604,30.68220,-88.06816,Mobile,AL,Alabama,True,NaN,9567,1398.1,...,False,Good,Good,Good,Good,Good,Suspicious,Good,Good,Good
32962,99753,64.93968,-161.15126,Koyuk,AK,Alaska,True,NaN,271,22.0,...,False,Bad,Good,Good,Good,Suspicious,Suspicious,Suspicious,Good,Good
32963,45871,40.49378,-84.29966,New Knoxville,OH,Ohio,True,NaN,2008,32.6,...,False,Good,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good
32964,99773,66.88758,-157.16496,Shungnak,AK,Alaska,True,NaN,288,18.6,...,False,Good,Good,Good,Good,Suspicious,Suspicious,Suspicious,Good,Good


In [2]:
import os
import shutil
import subprocess
import openstudio
import pandas as pd

def run_energyplus_simulations(year):
    input_csv = f'resources/zip_code_list_{year}.csv'
    # Load the input CSV.
    df = pd.read_csv(input_csv)

    # Add the "EnergyPlus Status" column if it doesn't exist.
    if "EnergyPlus Status" not in df.columns:
        df["EnergyPlus Status"] = ""

    # Get the unique weather station IDs.
    unique_stations = df[f"weather_station_wmo_{year}"].unique()

    # Set EnergyPlus installation location.
    energyplus_path = '/Applications/EnergyPlus-23-2-0/energyplus'
    current_directory = os.getcwd()

    print('***************')
    print(len(unique_stations))
    print('***************')

    k = 0
    for station in unique_stations:
        print(k)
        # If any row with this weather station already has a non-empty status, skip simulation.
        station_rows = df[df[f"weather_station_wmo_{year}"] == station]
        if station_rows["EnergyPlus Status"].notna().all() and (station_rows["EnergyPlus Status"] != "").all():
            continue

        # # Create (or reuse) the simulation directory.
        out_dir = "test_epws_dir"
        shutil.copy("resources/min.idf", "test_epws_dir/in.epw")
        epw_path = os.path.join(f"epws_wmo_{year}", f'{station}_{year}.epw')
        shutil.copy(epw_path, os.path.join(out_dir, "in.epw"))

        # Change directory to the simulation folder.
        os.chdir(out_dir)

        # Run EnergyPlus simulation.
        process = subprocess.run(
            [energyplus_path, "-w", "in.epw", "in.idf"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )

        # Check if the simulation ran successfully.
        status = "Good" if process.stderr.strip() == "EnergyPlus Completed Successfully." else "Bad"
        # if status == "Bad":
        #     print(f"Errors for {station}:\n{process.stderr}")

        # Return to the original directory.
        os.chdir(current_directory)

        # Update all rows corresponding to this weather station with the simulation status.
        df.loc[df[f"weather_station_wmo_{year}"] == station, "EnergyPlus Status"] = status
        df.to_csv(input_csv, index=False)
        # Reload the CSV before moving to the next station.
        df = pd.read_csv(input_csv)

        k+=1

    return df

year = 2024
# Run the EnergyPlus simulations and print the updated DataFrame.
final_df = run_energyplus_simulations(year)
final_df


***************
1541
***************
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0

,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,...,RH_holes_2024,EnergyPlus Status,Missing Data,Extreme Temperature,Sudden Temp Jumps,Constant Temp Periods,Day-Night Swings,Summer Freezing,Winter Extreme Heat,Missing Temperature Data
0,99638,52.89995,-168.93738,Nikolski,AK,Alaska,True,NaN,26,0.5,...,True,Good,Good,Good,Good,Suspicious,Good,Good,Good,Good
1,99923,55.97796,-130.03671,Hyder,AK,Alaska,True,NaN,15,0.4,...,False,Good,Good,Good,Suspicious,Suspicious,Suspicious,Good,Good,Good
2,99776,63.38825,-143.41140,Tanacross,AK,Alaska,True,NaN,141,2.6,...,False,Good,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good
3,99566,61.28097,-142.78996,Chitina,AK,Alaska,True,NaN,98,0.0,...,False,Good,Good,Good,Suspicious,Suspicious,Suspicious,Good,Good,Good
4,99780,63.19468,-143.07797,Tok,AK,Alaska,True,NaN,1559,0.5,...,False,Good,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23370,60302,41.89463,-87.78971,Oak Park,IL,Illinois,True,NaN,32048,4109.6,...,False,Good,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good
23371,79410,33.56969,-101.89127,Lubbock,TX,Texas,True,NaN,10160,1622.3,...,False,Good,Good,Good,Good,Good,Suspicious,Good,Good,Good
23372,76115,32.67879,-97.33076,Fort Worth,TX,Texas,True,NaN,20507,1699.2,...,False,Good,Good,Good,Good,Good,Suspicious,Good,Good,Good
23373,29684,34.38030,-82.72663,Starr,SC,South Carolina,True,NaN,3915,26.9,...,False,Good,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good


Delete files


In [3]:
import pandas as pd
import os

# Define paths: update these with your actual CSV file and folder path.
csv_file = 'resources/zip_code_list_2023.csv'           # Path to your CSV file
epw_folder = 'epws_wmo_2023'  # Folder where the .epw files are stored

# Read the CSV file
df = pd.read_csv(csv_file)

# Loop over the rows where "EnergyPlus Status" is "Bad"
for idx, row in df.iterrows():
    if row['EnergyPlus Status'].strip().lower() == 'bad':
        # Assuming there's a column named "FileName" that has the file names
        file_name = row['weather_station_wmo_2023'].strip()
        file_name = file_name + '_2023.epw'
        file_path = os.path.join(epw_folder, file_name)
        
        if os.path.exists(file_path):
            os.remove(file_path)
            print(f"Deleted: {file_path}")
        else:
            print(f"File not found: {file_path}")


Deleted: epws_wmo_2023/70489_2023.epw
Deleted: epws_wmo_2023/76061_2023.epw
Deleted: epws_wmo_2023/72665_2023.epw
Deleted: epws_wmo_2023/KPHP0_2023.epw
Deleted: epws_wmo_2023/KIDA0_2023.epw
Deleted: epws_wmo_2023/BN2KA_2023.epw
File not found: epws_wmo_2023/72665_2023.epw
Deleted: epws_wmo_2023/KCVS0_2023.epw
Deleted: epws_wmo_2023/KCVN0_2023.epw
Deleted: epws_wmo_2023/VF54M_2023.epw
File not found: epws_wmo_2023/BN2KA_2023.epw
File not found: epws_wmo_2023/BN2KA_2023.epw
Deleted: epws_wmo_2023/KMYL0_2023.epw
Deleted: epws_wmo_2023/8NL5S_2023.epw
Deleted: epws_wmo_2023/72580_2023.epw
Deleted: epws_wmo_2023/R521S_2023.epw
File not found: epws_wmo_2023/VF54M_2023.epw
Deleted: epws_wmo_2023/70259_2023.epw
Deleted: epws_wmo_2023/72622_2023.epw
Deleted: epws_wmo_2023/KLKV0_2023.epw
Deleted: epws_wmo_2023/KSRR0_2023.epw
Deleted: epws_wmo_2023/4X77V_2023.epw
File not found: epws_wmo_2023/KPHP0_2023.epw
Deleted: epws_wmo_2023/KONO0_2023.epw
Deleted: epws_wmo_2023/KFAM0_2023.epw
Deleted: epws_w

In [1]:
import os
import shutil
import subprocess
import openstudio
import pandas as pd
import numpy as np
import sys
import glob

def check_epw_quality(epw_path):
    """
    Perform quality checks on an EPW file and return a summary of whether each variable is 'Good' or 'Suspicious'.
    
    Parameters:
        epw_path (str): Path to the EPW file.
    
    Returns:
        dict: Dictionary with quality check results for each tested variable.
    """
    # Read EPW file (skip header, start from line 9)
    epw_df = pd.read_csv(epw_path, skiprows=8, header=None)

    # Dictionary to store check results
    quality_checks = {}

    ### 1️⃣ Check for Missing Data ###
    missing_values = epw_df.isnull().sum().sum()
    quality_checks["Missing Data"] = "Suspicious" if missing_values > 0 else "Good"

    ### 2️⃣ Enhanced Dry Bulb Temperature Checks ###
    dry_bulb_temp = epw_df[6]
    month = epw_df[1]

    # Extreme values (-50°C to 60°C)
    extreme_values = ((dry_bulb_temp < -50) | (dry_bulb_temp > 60)).any()
    quality_checks["Extreme Temperature"] = "Suspicious" if extreme_values else "Good"

    # Sudden jumps (> 15°C per hour)
    temp_diff = dry_bulb_temp.diff().abs()
    rapid_jumps = (temp_diff > 15).any()
    quality_checks["Sudden Temp Jumps"] = "Suspicious" if rapid_jumps else "Good"

    # Constant temperature for more than 12 hours
    const_periods = (dry_bulb_temp.rolling(window=12, min_periods=1).std() == 0).any()
    quality_checks["Constant Temp Periods"] = "Suspicious" if const_periods else "Good"

    # Unrealistic day-night swings (<3°C or >30°C)
    epw_df["daily_max"] = dry_bulb_temp.groupby(epw_df[2]).transform("max")
    epw_df["daily_min"] = dry_bulb_temp.groupby(epw_df[2]).transform("min")
    epw_df["daily_range"] = epw_df["daily_max"] - epw_df["daily_min"]
    unrealistic_swings = ((epw_df["daily_range"] < 3) | (epw_df["daily_range"] > 30)).any()
    quality_checks["Day-Night Swings"] = "Suspicious" if unrealistic_swings else "Good"

    # Seasonal temperature mismatches
    summer_issues = ((month.isin([6, 7, 8])) & (dry_bulb_temp < 0)).any()  # Summer months with freezing temps
    winter_issues = ((month.isin([12, 1, 2])) & (dry_bulb_temp > 40)).any()  # Winter months with extreme heat
    quality_checks["Summer Freezing"] = "Suspicious" if summer_issues else "Good"
    quality_checks["Winter Extreme Heat"] = "Suspicious" if winter_issues else "Good"

    # Missing temperature data
    missing_temps = dry_bulb_temp.isna().any()
    quality_checks["Missing Temperature Data"] = "Suspicious" if missing_temps else "Good"

    return quality_checks


def check_final_epws(year):
    # File paths for input and intermediate (processed) CSVs
    input_csv = f'resources/zip_code_list_{year}.csv'
    processed_csv = f'resources/zip_code_list_{year}_processed.csv'

    # If a processed file exists, load it; otherwise load the original file.
    if os.path.exists(processed_csv):
        zip_code_list = pd.read_csv(processed_csv)
    else:
        zip_code_list = pd.read_csv(input_csv)

    # Folder containing EPW files
    epw_folder = f"epws_wmo_{year}"
    # set energyplus installation location
    energyplus_path = '/Applications/EnergyPlus-23-2-0/energyplus'

    current_directory = os.getcwd()

    # Process each EPW file from the DataFrame
    for index, row in zip_code_list.iterrows():
        # Skip if EnergyPlus Status already exists and is non-empty.
        if pd.notna(row.get("EnergyPlus Status")) and row.get("EnergyPlus Status") != "":
            continue

        epw_filename = row[f"weather_station_wmo_{year}"]
        epw_path = os.path.join(epw_folder, f'{epw_filename}_{year}.epw')
        # Skip if file doesn't exist
        if not os.path.exists(epw_path):
            print(f"❌ File not found: {epw_path}. Skipping...")
            zip_code_list.at[index, "EnergyPlus Status"] = "File Not Found"
            # Save progress after each row update
            zip_code_list.to_csv(processed_csv, index=False)
            continue

        # Run EPW quality checks
        quality_checks = check_epw_quality(epw_path)

        # Add quality check results into the DataFrame
        for key, value in quality_checks.items():
            zip_code_list.at[index, key] = value

        # Silence OpenStudio logging while loading the IDF
        devnull = open(os.devnull, 'w')
        old_stdout, old_stderr = os.dup(1), os.dup(2)  # Save original stdout/stderr
        os.dup2(devnull.fileno(), 1)  # Redirect stdout to devnull
        os.dup2(devnull.fileno(), 2)  # Redirect stderr to devnull

        try:
            min_idf = openstudio.IdfFile().load("resources/min.idf").get()
        finally:
            os.dup2(old_stdout, 1)  # Restore stdout
            os.dup2(old_stderr, 2)  # Restore stderr
            try:
                devnull.close()  # Close devnull
            except RuntimeError:
                print(epw_path)

        # Define output directory
        out_dir = "test_epws_dir"
        os.makedirs(out_dir, exist_ok=True)  # Ensure the directory exists

        # Save IDF file
        min_idf.save(f"{out_dir}/in.idf", True)

        # Save EPW file
        shutil.copy(epw_path, f"{out_dir}/in.epw")

        # Change to IDF directory
        os.chdir(out_dir)

        # Run EnergyPlus as a subprocess
        process = subprocess.run(
            [energyplus_path, "-w", "in.epw", "in.idf"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,  # Ensure text output for easier processing
        )

        # Determine if EnergyPlus run was successful
        status = "Good" if process.stderr.strip() == "EnergyPlus Completed Successfully." else "Bad"

        # Store the EnergyPlus status in the DataFrame
        zip_code_list.at[index, "EnergyPlus Status"] = status

        # Print errors only if run failed
        if status == "Bad":
            print(f"Errors for {epw_filename}:\n{process.stderr}")

        # Change back to main directory
        os.chdir(current_directory)

        # Save progress after processing each EPW file.
        zip_code_list.to_csv(processed_csv, index=False)

    return zip_code_list


year = 2022
# Process the EPW files and print the final DataFrame with updated checks
final = check_final_epws(year)
final


KeyboardInterrupt: 

In [1]:
import os
import shutil
import subprocess
import openstudio
import pandas as pd
import numpy as np
import sys
import glob

def check_epw_quality(epw_path):
    """
    Perform quality checks on an EPW file and return a summary of whether each variable is 'Good' or 'Suspicious'.
    
    Parameters:
        epw_path (str): Path to the EPW file.
    
    Returns:
        dict: Dictionary with quality check results for each tested variable.
    """
    # Read EPW file (skip header, start from line 9)
    epw_df = pd.read_csv(epw_path, skiprows=8, header=None)

    # Dictionary to store check results
    quality_checks = {}

    ### 1️⃣ Check for Missing Data ###
    missing_values = epw_df.isnull().sum().sum()
    quality_checks["Missing Data"] = "Suspicious" if missing_values > 0 else "Good"

    ### 2️⃣ Enhanced Dry Bulb Temperature Checks ###
    dry_bulb_temp = epw_df[6]
    month = epw_df[1]

    # Extreme values (-50°C to 60°C)
    extreme_values = ((dry_bulb_temp < -50) | (dry_bulb_temp > 60)).any()
    quality_checks["Extreme Temperature"] = "Suspicious" if extreme_values else "Good"

    # Sudden jumps (> 15°C per hour)
    temp_diff = dry_bulb_temp.diff().abs()
    rapid_jumps = (temp_diff > 15).any()
    quality_checks["Sudden Temp Jumps"] = "Suspicious" if rapid_jumps else "Good"

    # Constant temperature for more than 12 hours
    const_periods = (dry_bulb_temp.rolling(window=12, min_periods=1).std() == 0).any()
    quality_checks["Constant Temp Periods"] = "Suspicious" if const_periods else "Good"

    # Unrealistic day-night swings (<3°C or >30°C)
    epw_df["daily_max"] = dry_bulb_temp.groupby(epw_df[2]).transform("max")
    epw_df["daily_min"] = dry_bulb_temp.groupby(epw_df[2]).transform("min")
    epw_df["daily_range"] = epw_df["daily_max"] - epw_df["daily_min"]
    unrealistic_swings = ((epw_df["daily_range"] < 3) | (epw_df["daily_range"] > 30)).any()
    quality_checks["Day-Night Swings"] = "Suspicious" if unrealistic_swings else "Good"

    # Seasonal temperature mismatches
    summer_issues = ((month.isin([6, 7, 8])) & (dry_bulb_temp < 0)).any()   # Summer months with freezing temps
    winter_issues = ((month.isin([12, 1, 2])) & (dry_bulb_temp > 40)).any() # Winter months with extreme heat
    quality_checks["Summer Freezing"] = "Suspicious" if summer_issues else "Good"
    quality_checks["Winter Extreme Heat"] = "Suspicious" if winter_issues else "Good"

    # Missing temperature data
    missing_temps = dry_bulb_temp.isna().any()
    quality_checks["Missing Temperature Data"] = "Suspicious" if missing_temps else "Good"

    return quality_checks


def check_final_epws(year):
    # File paths for input and intermediate (processed) CSVs
    input_csv = f'resources/zip_code_list_{year}.csv'
    processed_csv = f'resources/zip_code_list_{year}_processed.csv'

    # If a processed file exists, load it; otherwise load the original file.
    if os.path.exists(processed_csv):
        zip_code_list = pd.read_csv(processed_csv)
    else:
        zip_code_list = pd.read_csv(input_csv)

    # Folder containing EPW files
    epw_folder = f"epws_wmo_{year}"
    # set energyplus installation location
    energyplus_path = '/Applications/EnergyPlus-23-2-0/energyplus'

    current_directory = os.getcwd()

    # Process each EPW file from the DataFrame
    for index, row in zip_code_list.iterrows():
        status_value = row.get("EnergyPlus Status")

        # Only process rows where "EnergyPlus Status" is NaN or "Bad"
        # If it's anything else (e.g., "Good", "File Not Found"), skip it.
        if not (pd.isna(status_value) or status_value == "Bad"):
            continue

        epw_filename = row.get(f"weather_station_wmo_{year}")
        if not isinstance(epw_filename, str):
            # If there's no valid EPW filename, skip
            zip_code_list.at[index, "EnergyPlus Status"] = "Invalid EPW Filename"
            zip_code_list.to_csv(processed_csv, index=False)
            continue

        epw_path = os.path.join(epw_folder, f'{epw_filename}_{year}.epw')

        # Skip if file doesn't exist
        if not os.path.exists(epw_path):
            print(f"❌ File not found: {epw_path}. Skipping...")
            zip_code_list.at[index, "EnergyPlus Status"] = "File Not Found"
            # Save progress after each row update
            zip_code_list.to_csv(processed_csv, index=False)
            continue

        # Run EPW quality checks
        quality_checks = check_epw_quality(epw_path)

        # Add quality check results into the DataFrame
        for key, value in quality_checks.items():
            zip_code_list.at[index, key] = value

        # Silence OpenStudio logging while loading the IDF
        devnull = open(os.devnull, 'w')
        old_stdout, old_stderr = os.dup(1), os.dup(2)  # Save original stdout/stderr
        os.dup2(devnull.fileno(), 1)  # Redirect stdout to devnull
        os.dup2(devnull.fileno(), 2)  # Redirect stderr to devnull

        try:
            min_idf = openstudio.IdfFile().load("resources/min.idf").get()
        finally:
            # Restore stdout and stderr
            os.dup2(old_stdout, 1)
            os.dup2(old_stderr, 2)
            try:
                devnull.close()
            except RuntimeError:
                print(epw_path)

        # Define output directory
        out_dir = "test_epws_dir"
        os.makedirs(out_dir, exist_ok=True)  # Ensure the directory exists

        # Save IDF file
        min_idf.save(f"{out_dir}/in.idf", True)

        # Save EPW file
        shutil.copy(epw_path, f"{out_dir}/in.epw")

        # Change to IDF directory
        os.chdir(out_dir)

        # Run EnergyPlus as a subprocess
        process = subprocess.run(
            [energyplus_path, "-w", "in.epw", "in.idf"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,  # Ensure text output for easier processing
        )

        # Determine if EnergyPlus run was successful
        status = "Good" if process.stderr.strip() == "EnergyPlus Completed Successfully." else "Bad"

        # Store the EnergyPlus status in the DataFrame
        zip_code_list.at[index, "EnergyPlus Status"] = status

        # Print errors only if run failed
        if status == "Bad":
            print(f"Errors for {epw_filename}:\n{process.stderr}")

        # Change back to main directory
        os.chdir(current_directory)

        # Save progress after processing each EPW file.
        zip_code_list.to_csv(processed_csv, index=False)

    return zip_code_list


# Example usage:
year = 2022
# Process the EPW files and print the final DataFrame with updated checks
final = check_final_epws(year)
final


,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,...,RH_holes_2022,Missing Data,Extreme Temperature,Sudden Temp Jumps,Constant Temp Periods,Day-Night Swings,Summer Freezing,Winter Extreme Heat,Missing Temperature Data,EnergyPlus Status
0,99638,52.89995,-168.93738,Nikolski,AK,Alaska,True,NaN,26,0.5,...,False,Good,Good,Good,Suspicious,Good,Good,Good,Good,Good
1,99923,55.97796,-130.03671,Hyder,AK,Alaska,True,NaN,15,0.4,...,False,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good,Good
2,99776,63.38825,-143.41140,Tanacross,AK,Alaska,True,NaN,141,2.6,...,False,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good,Good
3,99566,61.28097,-142.78996,Chitina,AK,Alaska,True,NaN,98,0.0,...,False,Good,Good,Suspicious,Good,Suspicious,Good,Good,Good,Good
4,99780,63.19468,-143.07797,Tok,AK,Alaska,True,NaN,1559,0.5,...,False,Good,Good,Suspicious,Good,Suspicious,Good,Good,Good,Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12721,28775,35.01980,-83.32836,Scaly Mountain,NC,North Carolina,True,NaN,267,10.5,...,False,Good,Good,Good,Good,Suspicious,Good,Good,Good,Good
12722,70817,30.37601,-90.98090,Baton Rouge,LA,Louisiana,True,NaN,35142,527.9,...,False,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good,Good
12723,29123,33.76637,-81.25947,Pelion,SC,South Carolina,True,NaN,8525,47.6,...,False,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good,Good
12724,25067,38.19767,-81.42894,East Bank,WV,West Virginia,True,NaN,1184,84.4,...,False,Good,Good,Good,Suspicious,Suspicious,Good,Good,Good,Good
